# Models 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from itertools import product
import torch

In [2]:
# import data
train = pd.read_parquet('../data/model/final_train.parquet')
val = pd.read_parquet('../data/model/final_val.parquet')
test = pd.read_parquet('../data/model/final_test.parquet')

In [3]:
# check the min and max Date in each dataset
print("Train Date range:", train['Date'].min(), "to", train['Date'].max())
print("Validation Date range:", val['Date'].min(), "to", val['Date'].max())
print("Test Date range:", test['Date'].min(), "to", test['Date'].max())

Train Date range: 2016-01-04 00:00:00 to 2021-12-30 00:00:00
Validation Date range: 2022-01-03 00:00:00 to 2022-12-30 00:00:00
Test Date range: 2023-01-03 00:00:00 to 2023-12-29 00:00:00


In [4]:
X_train = train.loc[:, train.columns != "return_next_day"]
X_val = val.loc[:, val.columns != "return_next_day"]
X_test = test.loc[:, test.columns != "return_next_day"]
X_full = pd.concat([X_train, X_val, X_test], axis=0).reset_index(drop=True)

In [5]:
X_train.columns

Index(['Date', 'tic', 'Open', 'High', 'Low', 'Close', 'Volume',
       'sales_growth_qoq', 'sales_growth_ttm', 'asset_growth',
       ...
       'pca_emb_54', 'pca_emb_55', 'pca_emb_56', 'pca_emb_57', 'pca_emb_58',
       'pca_emb_59', 'pca_emb_60', 'pca_emb_61', 'pca_emb_62', 'pca_emb_63'],
      dtype='object', length=108)

In [6]:
y_train = train[['Date', 'tic', 'return_next_day']]
y_val = val[['Date', 'tic', 'return_next_day']]
y_test = test[['Date', 'tic', 'return_next_day']]
y_full = pd.concat([y_train, y_val, y_test], axis=0).reset_index(drop=True)

In [7]:
def directional_accuracy(y_true, y_pred):
    return (np.sign(y_true) == np.sign(y_pred)).mean()

#### Model 1 - using features from baseline 

From the baseline, we found a couple of features that are helpful to predict next day return 
- s4_dow_mean
- s7_roll63_mean_return
- ema_ema_50, ema_ema_20, ema_ema_10
- b0 (simple lag return)
- ma_1m, ma_3d, ma_14d, ma_30d, ma_50d, ma_2w, ma_2d, ma_1w
- s3_month_of_year_mean

In [8]:
def add_rolling_mean_feature(df, group_col, value_col, window, feature_name, shift_before=True):
    """
    Rolling mean feature per group_col.

    If shift_before=True, shift by 1 so feature at time t only uses info before t.
    """
    df = df.copy()
    df[feature_name] = (
        df.groupby(group_col)[value_col]
          .transform(lambda x: x.rolling(window=window, min_periods=1).mean())
    )
    if shift_before:
        df[feature_name] = df.groupby(group_col)[feature_name].shift(1)
    return df


def add_ema_feature(df, group_col, value_col, span, feature_name, shift_before=True):
    """
    Exponential moving average feature per group_col.

    If shift_before=True, shift by 1 so there is no look-ahead.
    """
    df = df.copy()
    df[feature_name] = (
        df.groupby(group_col)[value_col]
          .transform(lambda x: x.ewm(span=span, adjust=False, min_periods=1).mean())
    )
    if shift_before:
        df[feature_name] = df.groupby(group_col)[feature_name].shift(1)
    return df

In [9]:
def compute_signal_features_from_Xfull(X_full):
    """
    X_full must have at least: Date, tic, Close.
    We compute ret internally and build all the signal features.

    Returns a DataFrame with:
        ['Date', 'tic',
         's4_dow_mean',
         's7_roll63_mean_return',
         'ema_ema_50', 'ema_ema_20', 'ema_ema_10',
         'b0',
         'ma_1m', 'ma_3d', 'ma_14d', 'ma_30d', 'ma_50d',
         'ma_2w', 'ma_2d', 'ma_1w',
         's3_month_of_year_mean']
    """
    df = X_full[['Date', 'tic', 'Close']].copy()
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values(['tic', 'Date']).reset_index(drop=True)

    # --- realized daily return (today vs yesterday) ---
    df['ret'] = df.groupby('tic')['Close'].pct_change()

    # --- b0: previous day's return per ticker ---
    df['b0'] = df.groupby('tic')['ret'].shift(1)

    # --- s4_dow_mean: prior day's cross-sectional mean ret (market-like) ---
    # cross-sectional mean ret per day
    df['dow_mean_ret'] = df.groupby('Date')['ret'].transform('mean')
    # use previous day's dow_mean_ret for each ticker to avoid look-ahead
    df['s4_dow_mean'] = df.groupby('tic')['dow_mean_ret'].shift(1)

    # --- s7_roll63_mean_return: 63-day rolling mean of ret per ticker ---
    df = add_rolling_mean_feature(
        df,
        group_col='tic',
        value_col='ret',
        window=63,
        feature_name='s7_roll63_mean_return',
        shift_before=True,
    )

    # --- EMA features on ret per ticker ---
    df = add_ema_feature(
        df,
        group_col='tic',
        value_col='ret',
        span=50,
        feature_name='ema_ema_50',
        shift_before=True,
    )
    df = add_ema_feature(
        df,
        group_col='tic',
        value_col='ret',
        span=20,
        feature_name='ema_ema_20',
        shift_before=True,
    )
    df = add_ema_feature(
        df,
        group_col='tic',
        value_col='ret',
        span=10,
        feature_name='ema_ema_10',
        shift_before=True,
    )

    # --- Moving-average features on ret per ticker ---
    ma_windows = {
        'ma_2d': 2,
        'ma_3d': 3,
        'ma_14d': 14,
        'ma_1w': 5,   # ~1 week
        'ma_2w': 10,  # ~2 weeks
        'ma_1m': 21,  # ~1 month
        'ma_30d': 30,
        'ma_50d': 50,
    }

    for label, w in ma_windows.items():
        df = add_rolling_mean_feature(
            df,
            group_col='tic',
            value_col='ret',
            window=w,
            feature_name=label,
            shift_before=True,
        )

    # --- s3_month_of_year_mean: expanding month-of-year mean ret per (tic, month) ---
    df['month'] = df['Date'].dt.month
    df = df.sort_values(['tic', 'month', 'Date']).reset_index(drop=True)

    def _expanding_month_mean(s):
        # expanding mean of past values only
        return s.shift(1).expanding(min_periods=1).mean()

    df['s3_month_of_year_mean'] = (
        df.groupby(['tic', 'month'])['ret'].transform(_expanding_month_mean)
    )

    # Sort back by tic, Date
    df = df.sort_values(['tic', 'Date']).reset_index(drop=True)

    feature_cols = [
        's4_dow_mean',
        's7_roll63_mean_return',
        'ema_ema_50',
        'ema_ema_20',
        'ema_ema_10',
        'b0',
        'ma_1m',
        'ma_3d',
        'ma_14d',
        'ma_30d',
        'ma_50d',
        'ma_2w',
        'ma_2d',
        'ma_1w',
        's3_month_of_year_mean',
    ]

    # Fill NaNs in features with 0 (no historical info yet → neutral signal)
    df[feature_cols] = df[feature_cols].fillna(0.0)

    features_df = df[['Date', 'tic'] + feature_cols].copy()
    return features_df

In [10]:
features_full = compute_signal_features_from_Xfull(X_full)
features_full.head()

,Date,tic,s4_dow_mean,s7_roll63_mean_return,ema_ema_50,ema_ema_20,ema_ema_10,b0,ma_1m,ma_3d,ma_14d,ma_30d,ma_50d,ma_2w,ma_2d,ma_1w,s3_month_of_year_mean
0,2016-01-04,A,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2016-01-05,A,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,2016-01-06,A,0.002172,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440
3,2016-01-07,A,-0.015380,0.000499,-0.003131,-0.002690,-0.002008,0.004439,0.000499,0.000499,0.000499,0.000499,0.000499,0.000499,0.000499,0.000499,0.000499
4,2016-01-08,A,-0.024183,-0.013825,-0.004674,-0.006479,-0.009365,-0.042474,-0.013825,-0.013825,-0.013825,-0.013825,-0.013825,-0.013825,-0.019018,-0.013825,-0.013825


In [11]:
# Augment design matrices with signal features
X_train_aug = X_train.merge(features_full, on=['Date', 'tic'], how='left')
X_val_aug   = X_val.merge(features_full,   on=['Date', 'tic'], how='left')
X_test_aug  = X_test.merge(features_full,  on=['Date', 'tic'], how='left')

# Optional sanity checks
print("X_train:", X_train.shape, "->", X_train_aug.shape)
print("X_val:  ", X_val.shape,   "->", X_val_aug.shape)
print("X_test: ", X_test.shape,  "->", X_test_aug.shape)

# Check for missing values in new features (should be minor if at all)
X_train_aug.isna().mean().sort_values().tail(15)

X_train: (729659, 108) -> (729659, 123)
X_val:   (124747, 108) -> (124747, 123)
X_test:  (124727, 108) -> (124727, 123)


pca_emb_0                0.0
news_count               0.0
sum_sentiment            0.0
min_sentiment            0.0
max_sentiment            0.0
mean_sentiment           0.0
yield_spread_10y_2y      0.0
sp500                    0.0
vix                      0.0
aaa_yield                0.0
t3m                      0.0
t2y                      0.0
t10y                     0.0
retail_sales             0.0
s3_month_of_year_mean    0.0
dtype: float64

In [12]:
feature_cols = [
    's4_dow_mean',
    's7_roll63_mean_return',
    'ema_ema_50',
    'ema_ema_20',
    'ema_ema_10',
    'b0',
    'ma_1m',
    'ma_3d',
    'ma_14d',
    'ma_30d',
    'ma_50d',
    'ma_2w',
    'ma_2d',
    'ma_1w',
    's3_month_of_year_mean',
]

In [13]:
# columns we want to keep
keep_cols = ['Date', 'tic', 'Close'] + feature_cols

X_train_sig = X_train_aug[keep_cols].copy()
X_val_sig   = X_val_aug[keep_cols].copy()
X_test_sig  = X_test_aug[keep_cols].copy()

In [14]:
X_train_sig.head()

,Date,tic,Close,s4_dow_mean,s7_roll63_mean_return,ema_ema_50,ema_ema_20,ema_ema_10,b0,ma_1m,ma_3d,ma_14d,ma_30d,ma_50d,ma_2w,ma_2d,ma_1w,s3_month_of_year_mean
0,2016-01-04,A,37.636356,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2016-01-05,A,37.506878,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,2016-01-06,A,37.673370,0.002172,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440
3,2016-01-07,A,36.073215,-0.015380,0.000499,-0.003131,-0.002690,-0.002008,0.004439,0.000499,0.000499,0.000499,0.000499,0.000499,0.000499,0.000499,0.000499,0.000499
4,2016-01-08,A,35.693970,-0.024183,-0.013825,-0.004674,-0.006479,-0.009365,-0.042474,-0.013825,-0.013825,-0.013825,-0.013825,-0.013825,-0.013825,-0.019018,-0.013825,-0.013825


In [15]:
# Features
X_train_m = X_train_sig.drop(columns=["Date", "tic"])
X_val_m   = X_val_sig.drop(columns=["Date", "tic"])
X_test_m  = X_test_sig.drop(columns=["Date", "tic"])

# Targets
y_train_vec = y_train["return_next_day"]
y_val_vec   = y_val["return_next_day"]
y_test_vec  = y_test["return_next_day"]

In [16]:
lr = LinearRegression()
lr.fit(X_train_m, y_train_vec)

# predictions
pred_val_lr = lr.predict(X_val_m)
pred_test_lr = lr.predict(X_test_m)

# evaluation
print("Linear Regression:")
print("Val DA:", directional_accuracy(y_val_vec, pred_val_lr))

print("Test DA:", directional_accuracy(y_test_vec, pred_test_lr))

Linear Regression:
Val DA: 0.4866890586547171
Test DA: 0.4956905882447265


In [17]:
# optional but helpful: cast to float32
X_train_m_32 = X_train_m.astype("float32")
X_val_m_32   = X_val_m.astype("float32")
X_test_m_32  = X_test_m.astype("float32")

y_train_vec = y_train_vec.astype("float32")
y_val_vec   = y_val_vec.astype("float32")
y_test_vec  = y_test_vec.astype("float32")

xgb_model = xgb.XGBRegressor(
    n_estimators=2000,      # large cap; early stopping will cut it
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",     # fast CPU algorithm
    n_jobs=-1,              # use all CPU cores
    random_state=42,
    eval_metric="rmse",     # <-- move eval_metric here
)

xgb_model.fit(
    X_train_m_32,
    y_train_vec,
    eval_set=[(X_val_m_32, y_val_vec)],
    verbose=False,
)

# Use the best iteration found by early stopping
pred_val_xgb  = xgb_model.predict(X_val_m_32)
pred_test_xgb = xgb_model.predict(X_test_m_32)

In [18]:
print("XGBoost (baseline):")
print("Val DA: ", directional_accuracy(y_val_vec, pred_val_xgb))

print("Test DA: ", directional_accuracy(y_test_vec, pred_test_xgb))

XGBoost (baseline):
Val DA:  0.4877632327831531
Test DA:  0.5093203556567544


Finetune on xgboost 

In [19]:
param_grid = {
    "max_depth":        [3, 4, 5],
    "learning_rate":    [0.03, 0.05, 0.1],
    "n_estimators":     [200, 400, 800],
    "subsample":        [0.7, 0.9],
    "colsample_bytree": [0.7, 0.9],
}

results = []

for max_depth, lr, n_estimators, subsample, colsample in product(
    param_grid["max_depth"],
    param_grid["learning_rate"],
    param_grid["n_estimators"],
    param_grid["subsample"],
    param_grid["colsample_bytree"],
):
    params = {
        "max_depth": max_depth,
        "learning_rate": lr,
        "n_estimators": n_estimators,
        "subsample": subsample,
        "colsample_bytree": colsample,
    }

    model = xgb.XGBRegressor(
        **params,
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
    )

    model.fit(X_train_m_32, y_train_vec, verbose=False)

    val_pred = model.predict(X_val_m_32)
    val_da   = directional_accuracy(y_val_vec, val_pred)

    results.append({
        "max_depth": max_depth,
        "learning_rate": lr,
        "n_estimators": n_estimators,
        "subsample": subsample,
        "colsample_bytree": colsample,
        "val_DA": val_da,
    })

results_df = pd.DataFrame(results)
results_df_sorted = results_df.sort_values("val_DA", ascending=False)
results_df_sorted.head()

,max_depth,learning_rate,n_estimators,subsample,colsample_bytree,val_DA
74,5,0.03,200,0.9,0.7,0.503996
48,4,0.05,200,0.7,0.7,0.503547
54,4,0.05,400,0.9,0.7,0.502601
44,4,0.03,800,0.7,0.7,0.502537
14,3,0.05,200,0.9,0.7,0.502337


In [20]:
best = results_df_sorted.iloc[0]
best_params = {
    "max_depth":        int(best["max_depth"]),
    "learning_rate":    float(best["learning_rate"]),
    "n_estimators":     int(best["n_estimators"]),
    "subsample":        float(best["subsample"]),
    "colsample_bytree": float(best["colsample_bytree"]),
}
print("Best params:", best_params)

best_xgb = xgb.XGBRegressor(
    **best_params,
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
)

best_xgb.fit(X_train_m_32, y_train_vec, verbose=False)

pred_val_best  = best_xgb.predict(X_val_m_32)   # optional, just to inspect
pred_test_best = best_xgb.predict(X_test_m_32)

print("Val DA: ", directional_accuracy(y_val_vec,  pred_val_best))
print("Test DA:", directional_accuracy(y_test_vec, pred_test_best))

Best params: {'max_depth': 5, 'learning_rate': 0.03, 'n_estimators': 200, 'subsample': 0.9, 'colsample_bytree': 0.7}
Val DA:  0.5039960880822786
Test DA: 0.5185164399047519


Feature ablation

In [21]:
# inspect your columns once
feature_cols = list(X_train_m_32.columns)
feature_cols

['Close',
 's4_dow_mean',
 's7_roll63_mean_return',
 'ema_ema_50',
 'ema_ema_20',
 'ema_ema_10',
 'b0',
 'ma_1m',
 'ma_3d',
 'ma_14d',
 'ma_30d',
 'ma_50d',
 'ma_2w',
 'ma_2d',
 'ma_1w',
 's3_month_of_year_mean']

In [22]:
# base
base_cols = ["Close"]

# MA / EMA group (plus the long rolling mean s7)
ma_ema_cols = [
    c for c in feature_cols
    if c.startswith("ma_") or c.startswith("ema_") or c == "s7_roll63_mean_return"
]

# DOW + seasonality group
dow_season_cols = []
if "s4_dow_mean" in feature_cols:
    dow_season_cols.append("s4_dow_mean")
if "s3_month_of_year_mean" in feature_cols:
    dow_season_cols.append("s3_month_of_year_mean")

# simple lag return
b0_cols = ["b0"] if "b0" in feature_cols else []

print("MA/EMA group:", ma_ema_cols)
print("DOW/Seasonality group:", dow_season_cols)
print("Lag group:", b0_cols)

MA/EMA group: ['s7_roll63_mean_return', 'ema_ema_50', 'ema_ema_20', 'ema_ema_10', 'ma_1m', 'ma_3d', 'ma_14d', 'ma_30d', 'ma_50d', 'ma_2w', 'ma_2d', 'ma_1w']
DOW/Seasonality group: ['s4_dow_mean', 's3_month_of_year_mean']
Lag group: ['b0']


In [23]:
def run_xgb_da(feature_list, label):
    """
    Train XGBoost on a subset of features and report Val/Test DA.
    feature_list: list of column names to use.
    label: string name for printing / logging.
    """
    Xtr = X_train_m_32[feature_list]
    Xva = X_val_m_32[feature_list]
    Xte = X_test_m_32[feature_list]

    model = xgb.XGBRegressor(
        **best_params,
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
    )

    model.fit(Xtr, y_train_vec, verbose=False)

    val_pred  = model.predict(Xva)
    test_pred = model.predict(Xte)

    val_da  = directional_accuracy(y_val_vec,  val_pred)
    test_da = directional_accuracy(y_test_vec, test_pred)

    print(f"{label}:")
    print(f"  Val DA:  {val_da:.4f}")
    print(f"  Test DA: {test_da:.4f}")
    print()
    
    return val_da, test_da

In [24]:
full_cols = feature_cols  # all columns in X_train_m_32
run_xgb_da(full_cols, "FULL MODEL")

FULL MODEL:
  Val DA:  0.5040
  Test DA: 0.5185



(np.float64(0.5039960880822786), np.float64(0.5185164399047519))

In [25]:
ma_ema_only_cols = base_cols + ma_ema_cols
run_xgb_da(ma_ema_only_cols, "ONLY MA/EMA + Close")

ONLY MA/EMA + Close:
  Val DA:  0.4901
  Test DA: 0.5187



(np.float64(0.49011198666100186), np.float64(0.5186607550891146))

In [26]:
dow_season_only_cols = base_cols + dow_season_cols
run_xgb_da(dow_season_only_cols, "ONLY DOW + Seasonality + Close")

ONLY DOW + Seasonality + Close:
  Val DA:  0.4942
  Test DA: 0.5136



(np.float64(0.49424034245312515), np.float64(0.5136177411466644))

In [27]:
no_ma_ema_cols = [c for c in feature_cols if c not in ma_ema_cols]
run_xgb_da(no_ma_ema_cols, "FULL - MA/EMA")

FULL - MA/EMA:
  Val DA:  0.4962
  Test DA: 0.5163



(np.float64(0.496172252639342), np.float64(0.5163196420983428))

In [28]:
no_dow_season_cols = [c for c in feature_cols if c not in dow_season_cols]
run_xgb_da(no_dow_season_cols, "FULL - DOW/Seasonality")

FULL - DOW/Seasonality:
  Val DA:  0.4925
  Test DA: 0.5185



(np.float64(0.492500821663046), np.float64(0.5184843698637825))

In [29]:
feature_names = X_train_m_32.columns

In [30]:
# assumes best_xgb is the trained model
booster = best_xgb.get_booster()
importance_dict = booster.get_score(importance_type="gain")

fi_gain = pd.DataFrame({
    "feature": list(importance_dict.keys()),
    "gain": list(importance_dict.values())
}).sort_values("gain", ascending=False)

fi_gain

,feature,gain
7,ma_1m,0.201014
1,s4_dow_mean,0.193971
13,ma_2d,0.160944
9,ma_14d,0.124181
14,ma_1w,0.115281
6,b0,0.111594
12,ma_2w,0.093945
10,ma_30d,0.093069
4,ema_ema_20,0.091173
3,ema_ema_50,0.086391


In [31]:
ranked_features = [
    "ma_1m",
    "s4_dow_mean",
    "ma_2d",
    "ma_14d",
    "ma_1w",
    "b0",
    "ma_2w",
    "ma_30d",
    "ema_ema_20",
    "ema_ema_50",
    "ma_3d",
    "ema_ema_10",
    "s7_roll63_mean_return",
    "ma_50d",
    "s3_month_of_year_mean",
    "Close",
]

In [32]:
for N in [3, 5, 8, 10, 12, 15]:
    keep = ranked_features[:N]
    run_xgb_da(keep, f"TOP {N} FEATURES")

TOP 3 FEATURES:
  Val DA:  0.4934
  Test DA: 0.5177

TOP 5 FEATURES:
  Val DA:  0.5001
  Test DA: 0.5148

TOP 8 FEATURES:
  Val DA:  0.5002
  Test DA: 0.5132

TOP 10 FEATURES:
  Val DA:  0.4989
  Test DA: 0.5160

TOP 12 FEATURES:
  Val DA:  0.5015
  Test DA: 0.5124

TOP 15 FEATURES:
  Val DA:  0.5016
  Test DA: 0.5168



In [33]:
# ========================================
# Top-K feature model (K chosen by Val DA)
# ========================================

# 1. Choose K based on your ablation results
K = 15  # Top 15 had the highest Val DA ≈ 0.5016

# ranked_features is already sorted by importance (highest → lowest)
topK = ranked_features[:K]
print(f"Top {K} features:", topK)

# -----------------------------
# (A) Train on TRAIN only → Val DA (no leakage)
# -----------------------------
xgb_topK_train = xgb.XGBRegressor(
    **best_params,
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
)

xgb_topK_train.fit(X_train_m_32[topK], y_train_vec, verbose=False)

pred_val_topK = xgb_topK_train.predict(X_val_m_32[topK])
val_DA_topK   = directional_accuracy(y_val_vec, pred_val_topK)

print(f"Top-{K} Features (Train-only fit) — Val DA:", val_DA_topK)

# -----------------------------
# (B) Train on TRAIN+VAL → Test DA
# -----------------------------
X_trainval_topK = pd.concat(
    [X_train_m_32[topK], X_val_m_32[topK]],
    axis=0
)
y_trainval_vec = np.concatenate([y_train_vec, y_val_vec])

xgb_topK_trainval = xgb.XGBRegressor(
    **best_params,
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
)

xgb_topK_trainval.fit(X_trainval_topK, y_trainval_vec, verbose=False)

pred_test_topK = xgb_topK_trainval.predict(X_test_m_32[topK])
test_DA_topK   = directional_accuracy(y_test_vec, pred_test_topK)

print(f"Top-{K} Features (Train+Val fit) — Test DA:", test_DA_topK)

Top 15 features: ['ma_1m', 's4_dow_mean', 'ma_2d', 'ma_14d', 'ma_1w', 'b0', 'ma_2w', 'ma_30d', 'ema_ema_20', 'ema_ema_50', 'ma_3d', 'ema_ema_10', 's7_roll63_mean_return', 'ma_50d', 's3_month_of_year_mean']
Top-15 Features (Train-only fit) — Val DA: 0.5016152693050735
Top-15 Features (Train+Val fit) — Test DA: 0.5221483720445453


In [34]:
# Directional accuracy for each model

# Linear Regression
val_DA_lr   = directional_accuracy(y_val_vec,  pred_val_lr)
test_DA_lr  = directional_accuracy(y_test_vec, pred_test_lr)

# XGBoost (tuned) with baseline features
val_DA_xgb  = directional_accuracy(y_val_vec,  pred_val_best)
test_DA_xgb = directional_accuracy(y_test_vec, pred_test_best)

# XGBoost (tuned) with Top-K signal features
val_DA_topK  = directional_accuracy(y_val_vec,  pred_val_topK)   # train-only → val
test_DA_topK = directional_accuracy(y_test_vec, pred_test_topK) # train+val → test

# Summary table
results_models = pd.DataFrame([
    {
        "model": "Linear Regression - baseline features only",
        "val_DA": val_DA_lr,
        "test_DA": test_DA_lr,
    },
    {
        "model": "XGBoost (tuned) - baseline features only",
        "val_DA": val_DA_xgb,
        "test_DA": test_DA_xgb,
    },
    {
        "model": f"XGBoost (tuned) - top {K} signal baseline features",
        "val_DA": val_DA_topK,
        "test_DA": test_DA_topK,
    },
])

results_models

,model,val_DA,test_DA
0,Linear Regression - baseline features only,0.486689,0.495691
1,XGBoost (tuned) - baseline features only,0.503996,0.518516
2,XGBoost (tuned) - top 15 signal baseline features,0.501615,0.522148


#### Model 2 - Use just core stock features 
- Close
- logVolume
- Day of week 
- Month of year
- Sector
- Subsector

In [35]:
X_train.head()

,Date,tic,Open,High,Low,Close,Volume,sales_growth_qoq,sales_growth_ttm,asset_growth,...,pca_emb_54,pca_emb_55,pca_emb_56,pca_emb_57,pca_emb_58,pca_emb_59,pca_emb_60,pca_emb_61,pca_emb_62,pca_emb_63
0,2016-01-04,A,37.978592,38.098833,37.312624,37.636356,3287300,0.02071,-0.00247,-0.309482,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2016-01-05,A,37.673370,37.876861,37.312638,37.506878,2587200,0.02071,-0.00247,-0.309482,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,2016-01-06,A,37.220145,37.913860,37.044401,37.673370,2103600,0.02071,-0.00247,-0.309482,...,0.142439,-0.088256,0.401539,0.468688,0.084403,-0.750782,-0.323059,0.027356,-0.609138,-0.380578
3,2016-01-07,A,37.127663,37.136914,35.897475,36.073215,3504300,0.02071,-0.00247,-0.309482,...,-0.344860,-0.048228,0.058563,0.191014,0.074542,-0.062964,-0.121446,0.122892,-0.254481,0.042727
4,2016-01-08,A,36.276692,36.729917,35.582976,35.693970,3736700,0.02071,-0.00247,-0.309482,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [36]:
print(X_train.columns.to_list())

['Date', 'tic', 'Open', 'High', 'Low', 'Close', 'Volume', 'sales_growth_qoq', 'sales_growth_ttm', 'asset_growth', 'equity_growth', 'roa_ttm', 'roe_ttm', 'gross_margin_ttm', 'oper_margin_ttm', 'net_margin_ttm', 'log_mktcap', 'bm', 'earnings_yield', 'cf_yield', 'sales_yield', 'div_yield', 'leverage', 'current_ratio', 'cash_assets', 'accruals_ta', 'cpi', 'fedfunds', 'industrial_production', 'gdp', 'retail_sales', 'unemployment', 't10y', 't2y', 't3m', 'aaa_yield', 'vix', 'sp500', 'yield_spread_10y_2y', 'mean_sentiment', 'max_sentiment', 'min_sentiment', 'sum_sentiment', 'news_count', 'pca_emb_0', 'pca_emb_1', 'pca_emb_2', 'pca_emb_3', 'pca_emb_4', 'pca_emb_5', 'pca_emb_6', 'pca_emb_7', 'pca_emb_8', 'pca_emb_9', 'pca_emb_10', 'pca_emb_11', 'pca_emb_12', 'pca_emb_13', 'pca_emb_14', 'pca_emb_15', 'pca_emb_16', 'pca_emb_17', 'pca_emb_18', 'pca_emb_19', 'pca_emb_20', 'pca_emb_21', 'pca_emb_22', 'pca_emb_23', 'pca_emb_24', 'pca_emb_25', 'pca_emb_26', 'pca_emb_27', 'pca_emb_28', 'pca_emb_29', 'pc

In [37]:
sp500_info = pd.read_csv('../data/sp500_companies.csv')
sp500_info.head()

,ticker,company_name,sector,subsector,cik
0,MMM,3M,Industrials,Industrial Conglomerates,66740
1,AOS,A. O. Smith,Industrials,Building Products,91142
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,1800
3,ABBV,AbbVie,Health Care,Biotechnology,1551152
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,1467373


In [38]:
def add_time_and_volume_features(df):
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df["day_of_week"] = df["Date"].dt.dayofweek
    df["month"]       = df["Date"].dt.month
    df["log_volume"]  = np.log1p(df["Volume"])
    return df

In [39]:
# Create augmented versions instead
X_train_tv = add_time_and_volume_features(X_train)
X_val_tv   = add_time_and_volume_features(X_val)
X_test_tv  = add_time_and_volume_features(X_test)

In [40]:
# 1. Prepare sector/subsector lookup
sector_info = (
    sp500_info[["ticker", "sector", "subsector"]]
    .rename(columns={"ticker": "tic"})
)

# 2. Add sector + subsector to each split (starting from your time/volume-augmented versions)
X_train_tv_sec = X_train_tv.merge(sector_info, on="tic", how="left")
X_val_tv_sec   = X_val_tv.merge(sector_info,   on="tic", how="left")
X_test_tv_sec  = X_test_tv.merge(sector_info,  on="tic", how="left")

X_train_tv_sec.head()

,Date,tic,Open,High,Low,Close,Volume,sales_growth_qoq,sales_growth_ttm,asset_growth,...,pca_emb_59,pca_emb_60,pca_emb_61,pca_emb_62,pca_emb_63,day_of_week,month,log_volume,sector,subsector
0,2016-01-04,A,37.978592,38.098833,37.312624,37.636356,3287300,0.02071,-0.00247,-0.309482,...,0.000000,0.000000,0.000000,0.000000,0.000000,0,1,15.005577,Health Care,Life Sciences Tools & Services
1,2016-01-05,A,37.673370,37.876861,37.312638,37.506878,2587200,0.02071,-0.00247,-0.309482,...,0.000000,0.000000,0.000000,0.000000,0.000000,1,1,14.766087,Health Care,Life Sciences Tools & Services
2,2016-01-06,A,37.220145,37.913860,37.044401,37.673370,2103600,0.02071,-0.00247,-0.309482,...,-0.750782,-0.323059,0.027356,-0.609138,-0.380578,2,1,14.559161,Health Care,Life Sciences Tools & Services
3,2016-01-07,A,37.127663,37.136914,35.897475,36.073215,3504300,0.02071,-0.00247,-0.309482,...,-0.062964,-0.121446,0.122892,-0.254481,0.042727,3,1,15.069502,Health Care,Life Sciences Tools & Services
4,2016-01-08,A,36.276692,36.729917,35.582976,35.693970,3736700,0.02071,-0.00247,-0.309482,...,0.000000,0.000000,0.000000,0.000000,0.000000,4,1,15.133714,Health Care,Life Sciences Tools & Services


In [41]:
cat_cols = ["day_of_week", "month", "sector", "subsector"]

# 1) One-hot encode each split, overwriting the original variables
X_train_tv_sec = pd.get_dummies(X_train_tv_sec, columns=cat_cols, drop_first=True)
X_val_tv_sec   = pd.get_dummies(X_val_tv_sec,   columns=cat_cols, drop_first=True)
X_test_tv_sec  = pd.get_dummies(X_test_tv_sec,  columns=cat_cols, drop_first=True)

# 2) Align val/test columns to train's columns (any missing dummies → 0)
train_cols = X_train_tv_sec.columns

X_val_tv_sec  = X_val_tv_sec.reindex(columns=train_cols, fill_value=0)
X_test_tv_sec = X_test_tv_sec.reindex(columns=train_cols, fill_value=0)

In [42]:
def select_stock_structure_features(df):
    base_cols = ["Date", "tic", "Close", "log_volume"]
    
    dow_cols      = [c for c in df.columns if c.startswith("day_of_week_")]
    month_cols    = [c for c in df.columns if c.startswith("month_")]
    sector_cols   = [c for c in df.columns if c.startswith("sector_")]
    subsector_cols= [c for c in df.columns if c.startswith("subsector_")]
    
    keep_cols = base_cols + dow_cols + month_cols + sector_cols + subsector_cols
    return df[keep_cols].copy()

X_train_core = select_stock_structure_features(X_train_tv_sec)
X_val_core   = select_stock_structure_features(X_val_tv_sec)
X_test_core  = select_stock_structure_features(X_test_tv_sec)

In [43]:
X_train_core.head()

,Date,tic,Close,log_volume,day_of_week_1,day_of_week_2,day_of_week_3,day_of_week_4,month_2,month_3,...,subsector_Systems Software,subsector_Technology Distributors,"subsector_Technology Hardware, Storage & Peripherals",subsector_Telecom Tower REITs,subsector_Timber REITs,subsector_Tobacco,subsector_Trading Companies & Distributors,subsector_Transaction & Payment Processing Services,subsector_Water Utilities,subsector_Wireless Telecommunication Services
0,2016-01-04,A,37.636356,15.005577,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,2016-01-05,A,37.506878,14.766087,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,2016-01-06,A,37.673370,14.559161,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,2016-01-07,A,36.073215,15.069502,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,2016-01-08,A,35.693970,15.133714,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False


In [44]:
# Features to use for modeling = all columns except Date, tic
core_feature_cols = [c for c in X_train_core.columns if c not in ["Date", "tic"]]

X_train_core_m = X_train_core[core_feature_cols]
X_val_core_m   = X_val_core[core_feature_cols]
X_test_core_m  = X_test_core[core_feature_cols]

In [45]:
lr_core = LinearRegression()
lr_core.fit(X_train_core_m, y_train_vec)

# predictions with lr_core (not lr)
pred_val_lr_core  = lr_core.predict(X_val_core_m)
pred_test_lr_core = lr_core.predict(X_test_core_m)

print("Linear Regression (core features):")
print("Val DA:", directional_accuracy(y_val_vec,  pred_val_lr_core))
print("Test DA:", directional_accuracy(y_test_vec, pred_test_lr_core))

Linear Regression (core features):
Val DA: 0.4805005330789518
Test DA: 0.5131687605730917


XGBoost

In [46]:
# ==============================
# 1) Cast core matrices to float32
# ==============================

X_train_core_32 = X_train_core_m.astype("float32")
X_val_core_32   = X_val_core_m.astype("float32")
X_test_core_32  = X_test_core_m.astype("float32")

y_train_vec = y_train_vec.astype("float32")
y_val_vec   = y_val_vec.astype("float32")
y_test_vec  = y_test_vec.astype("float32")

# ==============================
# 2) XGBoost model on core features
# ==============================

xgb_model_core = xgb.XGBRegressor(
    n_estimators=2000,      # large cap; you can tune later
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
    eval_metric="rmse",
)

xgb_model_core.fit(
    X_train_core_32,
    y_train_vec,
    eval_set=[(X_val_core_32, y_val_vec)],
    verbose=False,
)


,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'rmse'


In [47]:
# ==============================
# 3) Predictions + DA
# ==============================

pred_val_xgb_core  = xgb_model_core.predict(X_val_core_32)
pred_test_xgb_core = xgb_model_core.predict(X_test_core_32)

print("XGBoost (core stock-structure features):")
print("Val DA:  ", directional_accuracy(y_val_vec,  pred_val_xgb_core))
print("Test DA: ", directional_accuracy(y_test_vec, pred_test_xgb_core))

XGBoost (core stock-structure features):
Val DA:   0.5136877039127193
Test DA:  0.519101718152445


Finetune XGBoost

In [48]:
param_grid_core = {
    "max_depth":        [3, 4, 5],
    "learning_rate":    [0.03, 0.05, 0.1],
    "n_estimators":     [200, 400, 800],
    "subsample":        [0.7, 0.9],
    "colsample_bytree": [0.7, 0.9],
}

results_core = []

for max_depth, lr, n_estimators, subsample, colsample in product(
    param_grid_core["max_depth"],
    param_grid_core["learning_rate"],
    param_grid_core["n_estimators"],
    param_grid_core["subsample"],
    param_grid_core["colsample_bytree"],
):
    params = {
        "max_depth": max_depth,
        "learning_rate": lr,
        "n_estimators": n_estimators,
        "subsample": subsample,
        "colsample_bytree": colsample,
    }

    model = xgb.XGBRegressor(
        **params,
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
    )

    model.fit(X_train_core_32, y_train_vec, verbose=False)

    val_pred = model.predict(X_val_core_32)
    val_da   = directional_accuracy(y_val_vec, val_pred)

    results_core.append({
        "max_depth": max_depth,
        "learning_rate": lr,
        "n_estimators": n_estimators,
        "subsample": subsample,
        "colsample_bytree": colsample,
        "val_DA": val_da,
    })

results_core_df = pd.DataFrame(results_core)
results_core_df_sorted = results_core_df.sort_values("val_DA", ascending=False)
results_core_df_sorted.head()

,max_depth,learning_rate,n_estimators,subsample,colsample_bytree,val_DA
65,4,0.1,400,0.7,0.9,0.516622
32,3,0.1,800,0.7,0.7,0.516525
33,3,0.1,800,0.7,0.9,0.516357
67,4,0.1,400,0.9,0.9,0.516149
62,4,0.1,200,0.9,0.7,0.516077


In [49]:
# Pick best params from CORE tuning results
best_core = results_core_df_sorted.iloc[0]
best_params_core = {
    "max_depth":        int(best_core["max_depth"]),
    "learning_rate":    float(best_core["learning_rate"]),
    "n_estimators":     int(best_core["n_estimators"]),
    "subsample":        float(best_core["subsample"]),
    "colsample_bytree": float(best_core["colsample_bytree"]),
}
print("Best CORE params:", best_params_core)

# XGBoost with best core params on core feature set
best_xgb_core = xgb.XGBRegressor(
    **best_params_core,
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
)

best_xgb_core.fit(X_train_core_32, y_train_vec, verbose=False)

pred_val_best_core  = best_xgb_core.predict(X_val_core_32)
pred_test_best_core = best_xgb_core.predict(X_test_core_32)

print("CORE XGBoost (best params):")
print("Val DA: ", directional_accuracy(y_val_vec,  pred_val_best_core))
print("Test DA:", directional_accuracy(y_test_vec, pred_test_best_core))

Best CORE params: {'max_depth': 4, 'learning_rate': 0.1, 'n_estimators': 400, 'subsample': 0.7, 'colsample_bytree': 0.9}
CORE XGBoost (best params):
Val DA:  0.5166216422038206
Test DA: 0.5213867085715203


In [50]:
feature_core_names = X_train_core_32.columns

In [51]:
# assumes best_xgb is the trained model
booster_core = best_xgb_core.get_booster()
importance_core_dict = booster_core.get_score(importance_type="gain")

fi_gain_core = pd.DataFrame({
    "feature": list(importance_core_dict.keys()),
    "gain": list(importance_core_dict.values())
}).sort_values("gain", ascending=False)

fi_gain_core

,feature,gain
2,day_of_week_1,0.022927
31,"subsector_Apparel, Accessories & Luxury Goods",0.020546
5,day_of_week_4,0.019373
27,subsector_Aerospace & Defense,0.018799
6,month_2,0.016777
...,...,...
28,subsector_Agricultural & Farm Machinery,0.001185
86,subsector_Industrial REITs,0.001124
49,subsector_Construction Machinery & Heavy Trans...,0.001110
138,subsector_Water Utilities,0.000998


In [52]:
# Compute DA for the core-feature models (using existing predictions)

val_DA_lr_core   = directional_accuracy(y_val_vec,  pred_val_lr_core)
test_DA_lr_core  = directional_accuracy(y_test_vec, pred_test_lr_core)

val_DA_xgb_core  = directional_accuracy(y_val_vec,  pred_val_best_core)
test_DA_xgb_core = directional_accuracy(y_test_vec, pred_test_best_core)

# Build new rows for the core models
core_rows = pd.DataFrame([
    {
        "model": "Linear Regression - core stock structure features",
        "val_DA": val_DA_lr_core,
        "test_DA": test_DA_lr_core,
    },
    {
        "model": "XGBoost (tuned) - core stock structure features",
        "val_DA": val_DA_xgb_core,
        "test_DA": test_DA_xgb_core,
    },
])

In [53]:
# Append to existing results_models (no baseline recompute)
results_models = pd.concat([results_models, core_rows], ignore_index=True)

results_models.sort_values('val_DA', ascending = False)

,model,val_DA,test_DA
4,XGBoost (tuned) - core stock structure features,0.516622,0.521387
1,XGBoost (tuned) - baseline features only,0.503996,0.518516
2,XGBoost (tuned) - top 15 signal baseline features,0.501615,0.522148
0,Linear Regression - baseline features only,0.486689,0.495691
3,Linear Regression - core stock structure features,0.480501,0.513169


#### Model 3 - Incorporate features from both Model 1 and 2 

In [54]:
# Drop Close from core so we keep the one from X_*_sig
X_train_sig_core = X_train_sig.merge(
    X_train_core.drop(columns=["Close"]),
    on=["Date", "tic"],
    how="left",
)

X_val_sig_core = X_val_sig.merge(
    X_val_core.drop(columns=["Close"]),
    on=["Date", "tic"],
    how="left",
)

X_test_sig_core = X_test_sig.merge(
    X_test_core.drop(columns=["Close"]),
    on=["Date", "tic"],
    how="left",
)

In [55]:
feature_cols_all = [c for c in X_train_sig_core.columns if c not in ["Date", "tic"]]

X_train_sig_core_m = X_train_sig_core[feature_cols_all]
X_val_sig_core_m   = X_val_sig_core[feature_cols_all]
X_test_sig_core_m  = X_test_sig_core[feature_cols_all]

In [56]:
# Linear Regression on signal + core stock-structure features
lr_sig_core = LinearRegression()
lr_sig_core.fit(X_train_sig_core_m, y_train_vec)

# predictions
pred_val_lr_sig_core  = lr_sig_core.predict(X_val_sig_core_m)
pred_test_lr_sig_core = lr_sig_core.predict(X_test_sig_core_m)

print("Linear Regression (signal + core stock-structure features):")
print("Val DA:",  directional_accuracy(y_val_vec,  pred_val_lr_sig_core))
print("Test DA:", directional_accuracy(y_test_vec, pred_test_lr_sig_core))

Linear Regression (signal + core stock-structure features):
Val DA: 0.4857992576975799
Test DA: 0.5028983299526165


XGBoost 

In [57]:
# ==============================
# 1) Cast matrices to float32
# ==============================

X_train_sig_core_32 = X_train_sig_core_m.astype("float32")
X_val_sig_core_32   = X_val_sig_core_m.astype("float32")
X_test_sig_core_32  = X_test_sig_core_m.astype("float32")

y_train_vec = y_train_vec.astype("float32")
y_val_vec   = y_val_vec.astype("float32")
y_test_vec  = y_test_vec.astype("float32")

# ==============================
# 2) XGBoost model on signal + core features
# ==============================

xgb_model_sig_core = xgb.XGBRegressor(
    n_estimators=2000,      # you can tune later
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
)

xgb_model_sig_core.fit(
    X_train_sig_core_32,
    y_train_vec,
    eval_set=[(X_val_sig_core_32, y_val_vec)],
    verbose=False,
)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [58]:
# ==============================
# 3) Predictions + DA
# ==============================

pred_val_xgb_sig_core  = xgb_model_sig_core.predict(X_val_sig_core_32)
pred_test_xgb_sig_core = xgb_model_sig_core.predict(X_test_sig_core_32)

print("XGBoost (signal + core stock-structure features):")
print("Val DA:  ", directional_accuracy(y_val_vec,  pred_val_xgb_sig_core))
print("Test DA: ", directional_accuracy(y_test_vec, pred_test_xgb_sig_core))

XGBoost (signal + core stock-structure features):
Val DA:   0.4941361315302172
Test DA:  0.5149967529083519


Finetune XGBoost

In [59]:
# =========================================
# 1) Hyperparameter grid for sig+core model
# =========================================

param_grid_sigcore = {
    "max_depth":        [3, 4, 5],
    "learning_rate":    [0.03, 0.05, 0.1],
    "n_estimators":     [200, 400, 800],
    "subsample":        [0.7, 0.9],
    "colsample_bytree": [0.7, 0.9],
}

results_sigcore = []

for max_depth, lr, n_estimators, subsample, colsample in product(
    param_grid_sigcore["max_depth"],
    param_grid_sigcore["learning_rate"],
    param_grid_sigcore["n_estimators"],
    param_grid_sigcore["subsample"],
    param_grid_sigcore["colsample_bytree"],
):
    params = {
        "max_depth": max_depth,
        "learning_rate": lr,
        "n_estimators": n_estimators,
        "subsample": subsample,
        "colsample_bytree": colsample,
    }

    model = xgb.XGBRegressor(
        **params,
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
    )

    # IMPORTANT: use sig+core matrices here
    model.fit(X_train_sig_core_32, y_train_vec, verbose=False)

    val_pred = model.predict(X_val_sig_core_32)
    val_da   = directional_accuracy(y_val_vec, val_pred)

    results_sigcore.append({
        "max_depth": max_depth,
        "learning_rate": lr,
        "n_estimators": n_estimators,
        "subsample": subsample,
        "colsample_bytree": colsample,
        "val_DA": val_da,
    })

results_sigcore_df = pd.DataFrame(results_sigcore)
results_sigcore_df_sorted = results_sigcore_df.sort_values("val_DA", ascending=False)
results_sigcore_df_sorted.head()

,max_depth,learning_rate,n_estimators,subsample,colsample_bytree,val_DA
47,4,0.03,800,0.9,0.9,0.513648
25,3,0.10,200,0.7,0.9,0.512998
55,4,0.05,400,0.9,0.9,0.512694
27,3,0.10,200,0.9,0.9,0.512469
97,5,0.10,200,0.7,0.9,0.512253


In [60]:
# =========================================
# 2) Pick best params and refit on sig+core
# =========================================

best_sigcore = results_sigcore_df_sorted.iloc[0]
best_params_sigcore = {
    "max_depth":        int(best_sigcore["max_depth"]),
    "learning_rate":    float(best_sigcore["learning_rate"]),
    "n_estimators":     int(best_sigcore["n_estimators"]),
    "subsample":        float(best_sigcore["subsample"]),
    "colsample_bytree": float(best_sigcore["colsample_bytree"]),
}
print("Best SIG+CORE params:", best_params_sigcore)

best_xgb_sigcore = xgb.XGBRegressor(
    **best_params_sigcore,
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
)

best_xgb_sigcore.fit(X_train_sig_core_32, y_train_vec, verbose=False)

pred_val_best_sigcore  = best_xgb_sigcore.predict(X_val_sig_core_32)
pred_test_best_sigcore = best_xgb_sigcore.predict(X_test_sig_core_32)

print("SIG+CORE XGBoost (best params):")
print("Val DA: ", directional_accuracy(y_val_vec,  pred_val_best_sigcore))
print("Test DA:", directional_accuracy(y_test_vec, pred_test_best_sigcore))

Best SIG+CORE params: {'max_depth': 4, 'learning_rate': 0.03, 'n_estimators': 800, 'subsample': 0.9, 'colsample_bytree': 0.9}
SIG+CORE XGBoost (best params):
Val DA:  0.5136476227885239
Test DA: 0.5139224065358744


In [61]:
feature_sigcore_names = X_train_sig_core_32.columns

In [62]:
# assumes best_xgb is the trained model
booster_sigcore = best_xgb_sigcore.get_booster()
importance_sigcore_dict = booster_sigcore.get_score(importance_type="gain")

fi_gain_sigcore = pd.DataFrame({
    "feature": list(importance_sigcore_dict.keys()),
    "gain": list(importance_sigcore_dict.values())
}).sort_values("gain", ascending=False)

fi_gain_sigcore

,feature,gain
22,month_3,0.282946
19,day_of_week_3,0.267327
20,day_of_week_4,0.205988
18,day_of_week_2,0.171212
17,day_of_week_1,0.140199
...,...,...
59,subsector_Health Care Distributors,0.003625
36,sector_Health Care,0.002579
89,"subsector_Technology Hardware, Storage & Perip...",0.002475
78,subsector_Pharmaceuticals,0.001494


In [63]:
sig_cols = feature_cols                                # price-signal features
vol_cols = ["log_volume"]
dow_cols = [c for c in X_train_sig_core_32.columns if c.startswith("day_of_week_")]
month_cols = [c for c in X_train_sig_core_32.columns if c.startswith("month_")]
sector_cols = [c for c in X_train_sig_core_32.columns if c.startswith("sector_")]
subsector_cols = [c for c in X_train_sig_core_32.columns if c.startswith("subsector_")]

In [64]:
def run_ablation(feature_list, label):
    """
    Train an XGBoost regressor on a subset of features and
    report BOTH validation and test directional accuracy.

    Uses:
      - X_train_sig_core_32, X_val_sig_core_32, X_test_sig_core_32
      - y_train_vec, y_val_vec, y_test_vec
      - best_params_sigcore
    """
    # subset columns
    Xtr = X_train_sig_core_32[feature_list]
    Xva = X_val_sig_core_32[feature_list]
    Xte = X_test_sig_core_32[feature_list]

    model = xgb.XGBRegressor(
        **best_params_sigcore,   # from your sig+core tuning
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
    )

    model.fit(Xtr, y_train_vec, verbose=False)

    # predictions
    val_pred  = model.predict(Xva)
    test_pred = model.predict(Xte)

    # directional accuracy
    val_da  = directional_accuracy(y_val_vec,  val_pred)
    test_da = directional_accuracy(y_test_vec, test_pred)

    print(f"{label}:")
    print(f"  Val DA:  {val_da:.4f}")
    print(f"  Test DA: {test_da:.4f}")
    print()

    return val_da, test_da

In [65]:
run_ablation(sig_cols, "Only price signals")

Only price signals:
  Val DA:  0.5014
  Test DA: 0.5145



(np.float64(0.5013908150095794), np.float64(0.5144756147425978))

In [66]:
run_ablation(sig_cols + vol_cols + dow_cols + month_cols, "Signals + time/volume")

Signals + time/volume:
  Val DA:  0.5061
  Test DA: 0.5185



(np.float64(0.5060803065404379), np.float64(0.5185084223945096))

In [67]:
run_ablation(sig_cols + sector_cols, "Signals + sector")

Signals + sector:
  Val DA:  0.4988
  Test DA: 0.5117



(np.float64(0.4988336392859147), np.float64(0.5116935386884957))

In [68]:
run_ablation(sig_cols + subsector_cols, "Signals + subsector")

Signals + subsector:
  Val DA:  0.5002
  Test DA: 0.5147



(np.float64(0.5002124299582354), np.float64(0.5147241575601113))

In [69]:
run_ablation(dow_cols + month_cols + sector_cols + subsector_cols, "No price signals")

No price signals:
  Val DA:  0.5140
  Test DA: 0.5221



(np.float64(0.513976288006926), np.float64(0.5220842319626063))

In [70]:
run_ablation(X_train_sig_core_32.columns, "Full model")

Full model:
  Val DA:  0.5136
  Test DA: 0.5139



(np.float64(0.5136476227885239), np.float64(0.5139224065358744))

Try different varieties - each column list is treated as a single group

In [71]:
from itertools import combinations

# --- Group dictionary: name -> list of columns ---
group_to_cols = {
    "sig":       sig_cols,
    "vol":       vol_cols,
    "dow":       dow_cols,
    "month":     month_cols,
    "sector":    sector_cols,
    "subsector": subsector_cols,
}

group_names = list(group_to_cols.keys())

def run_ablation(feature_list, label):
    """
    Train an XGBoost regressor on a subset of features and
    report BOTH validation and test directional accuracy.

    Uses:
      - X_train_sig_core_32, X_val_sig_core_32, X_test_sig_core_32
      - y_train_vec, y_val_vec, y_test_vec
      - best_params_sigcore
    """
    Xtr = X_train_sig_core_32[feature_list]
    Xva = X_val_sig_core_32[feature_list]
    Xte = X_test_sig_core_32[feature_list]

    model = xgb.XGBRegressor(
        **best_params_sigcore,
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
    )

    model.fit(Xtr, y_train_vec, verbose=False)

    val_pred  = model.predict(Xva)
    test_pred = model.predict(Xte)

    val_da  = directional_accuracy(y_val_vec,  val_pred)
    test_da = directional_accuracy(y_test_vec, test_pred)

    print(f"{label}:")
    print(f"  #features: {len(feature_list)}")
    print(f"  Val DA:    {val_da:.4f}")
    print(f"  Test DA:   {test_da:.4f}")
    print()

    return val_da, test_da

In [72]:
# ==========================================
# Systematic group ablation over all combos
# ==========================================
ablation_results = []

# r = number of groups in the combo (1..6)
for r in range(1, len(group_names) + 1):
    for combo in combinations(group_names, r):
        combo = list(combo)
        label = "+".join(combo)

        # union of columns from all groups in this combo
        cols = []
        for g in combo:
            cols.extend(group_to_cols[g])
        # dedupe and sort for stability
        cols = sorted(set(cols))

        val_da, test_da = run_ablation(cols, label)

        ablation_results.append({
            "groups": label,
            "num_features": len(cols),
            "val_DA": val_da,
            "test_DA": test_da,
        })

sig:
  #features: 16
  Val DA:    0.5008
  Test DA:   0.5132

vol:
  #features: 1
  Val DA:    0.4926
  Test DA:   0.5194

dow:
  #features: 4
  Val DA:    0.4931
  Test DA:   0.5199

month:
  #features: 11
  Val DA:    0.4806
  Test DA:   0.5117

sector:
  #features: 10
  Val DA:    0.4931
  Test DA:   0.5199

subsector:
  #features: 124
  Val DA:    0.4931
  Test DA:   0.5197

sig+vol:
  #features: 17
  Val DA:    0.4998
  Test DA:   0.5120

sig+dow:
  #features: 20
  Val DA:    0.5028
  Test DA:   0.5040

sig+month:
  #features: 27
  Val DA:    0.5145
  Test DA:   0.5192

sig+sector:
  #features: 26
  Val DA:    0.4984
  Test DA:   0.5121

sig+subsector:
  #features: 140
  Val DA:    0.5011
  Test DA:   0.5118

vol+dow:
  #features: 5
  Val DA:    0.4958
  Test DA:   0.5175

vol+month:
  #features: 12
  Val DA:    0.4884
  Test DA:   0.5195

vol+sector:
  #features: 11
  Val DA:    0.4921
  Test DA:   0.5190

vol+subsector:
  #features: 125
  Val DA:    0.4937
  Test DA:   0.5196

d

In [73]:
ablation_df = pd.DataFrame(ablation_results).sort_values("val_DA", ascending=False)
ablation_df

,groups,num_features,val_DA,test_DA
37,dow+month+sector,25,0.514602,0.528145
22,sig+vol+month,28,0.514497,0.520328
8,sig+month,27,0.514497,0.519214
61,vol+dow+month+sector+subsector,150,0.514297,0.524016
55,dow+month+sector+subsector,149,0.513752,0.522750
...,...,...,...,...
54,vol+month+sector+subsector,146,0.487587,0.517859
18,month+sector,21,0.486561,0.517017
40,month+sector+subsector,145,0.484541,0.515911
19,month+subsector,135,0.483002,0.515269


Each signal is treated differently while including all other groups

In [74]:
# --- Groups already defined earlier ---
sig_cols = feature_cols                                # price-signal features
vol_cols = ["log_volume"]
dow_cols = [c for c in X_train_sig_core_32.columns if c.startswith("day_of_week_")]
month_cols = [c for c in X_train_sig_core_32.columns if c.startswith("month_")]
sector_cols = [c for c in X_train_sig_core_32.columns if c.startswith("sector_")]
subsector_cols = [c for c in X_train_sig_core_32.columns if c.startswith("subsector_")]

# Structural features we always keep
base_struct_cols = dow_cols + month_cols + sector_cols 

def run_ablation(feature_list, label):
    """
    Train an XGBoost regressor on a subset of features and
    report BOTH validation and test directional accuracy.

    Uses:
      - X_train_sig_core_32, X_val_sig_core_32, X_test_sig_core_32
      - y_train_vec, y_val_vec, y_test_vec
      - best_params_sigcore
    """
    Xtr = X_train_sig_core_32[feature_list]
    Xva = X_val_sig_core_32[feature_list]
    Xte = X_test_sig_core_32[feature_list]

    model = xgb.XGBRegressor(
        **best_params_sigcore,
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
    )

    model.fit(Xtr, y_train_vec, verbose=False)

    val_pred  = model.predict(Xva)
    test_pred = model.predict(Xte)

    val_da  = directional_accuracy(y_val_vec,  val_pred)
    test_da = directional_accuracy(y_test_vec, test_pred)

    print(f"{label}:")
    print(f"  #features: {len(feature_list)}")
    print(f"  Val DA:    {val_da:.4f}")
    print(f"  Test DA:   {test_da:.4f}")
    print()

    return val_da, test_da

In [81]:
# ranked_features is your ordered list of signal columns, e.g.
# ranked_features = ["ma_1m", "s4_dow_mean", "ma_2d", ...]
ablation_sig_results = []

for K in range(len(ranked_features)):
    sig_subset = ranked_features[:K] if K > 0 else []
    feature_list = base_struct_cols + sig_subset
    label = f"STRUCT + top{K}_signals" if K > 0 else "STRUCT only (no signals)"

    val_da, test_da = run_ablation(feature_list, label)
    ablation_sig_results.append({
        "K_signals": K,
        "label": label,
        "num_features": len(feature_list),
        "val_DA": val_da,
        "test_DA": test_da,
    })

STRUCT only (no signals):
  #features: 25
  Val DA:    0.5144
  Test DA:   0.5280

STRUCT + top1_signals:
  #features: 26
  Val DA:    0.5163
  Test DA:   0.5149

STRUCT + top2_signals:
  #features: 27
  Val DA:    0.4976
  Test DA:   0.5181

STRUCT + top3_signals:
  #features: 28
  Val DA:    0.4852
  Test DA:   0.5134

STRUCT + top4_signals:
  #features: 29
  Val DA:    0.5028
  Test DA:   0.5167

STRUCT + top5_signals:
  #features: 30
  Val DA:    0.4982
  Test DA:   0.5161

STRUCT + top6_signals:
  #features: 31
  Val DA:    0.5048
  Test DA:   0.5062

STRUCT + top7_signals:
  #features: 32
  Val DA:    0.5004
  Test DA:   0.5176

STRUCT + top8_signals:
  #features: 33
  Val DA:    0.4944
  Test DA:   0.5125

STRUCT + top9_signals:
  #features: 34
  Val DA:    0.5021
  Test DA:   0.5148

STRUCT + top10_signals:
  #features: 35
  Val DA:    0.5071
  Test DA:   0.5165

STRUCT + top11_signals:
  #features: 36
  Val DA:    0.5043
  Test DA:   0.5126

STRUCT + top12_signals:
  #features

In [82]:
ablation_sig_df = pd.DataFrame(ablation_sig_results).sort_values("val_DA", ascending=False)
ablation_sig_df

,K_signals,label,num_features,val_DA,test_DA
1,1,STRUCT + top1_signals,26,0.516261,0.514909
0,0,STRUCT only (no signals),25,0.514425,0.527969
13,13,STRUCT + top13_signals,38,0.510161,0.514171
14,14,STRUCT + top14_signals,39,0.507419,0.511750
15,15,STRUCT + top15_signals,40,0.507251,0.518067
10,10,STRUCT + top10_signals,35,0.507066,0.516512
6,6,STRUCT + top6_signals,31,0.504766,0.506242
11,11,STRUCT + top11_signals,36,0.504309,0.512624
4,4,STRUCT + top4_signals,29,0.502802,0.516737
9,9,STRUCT + top9_signals,34,0.502104,0.514812


In [85]:
# ---- 1. DA for LR + XGB on (signal + core) ----
val_DA_lr_sig_core   = directional_accuracy(y_val_vec,  pred_val_lr_sig_core)
test_DA_lr_sig_core  = directional_accuracy(y_test_vec, pred_test_lr_sig_core)

val_DA_xgb_sig_core  = directional_accuracy(y_val_vec,  pred_val_best_sigcore)
test_DA_xgb_sig_core = directional_accuracy(y_test_vec, pred_test_best_sigcore)

# ---- 2. Best models from the two ablation tables ----
best_ablation     = ablation_df.iloc[0]      # from group ablation (signals vs core groups)
best_ablation_sig = ablation_sig_df.iloc[0]  # from signal-only K-ablation

new_rows = [
    {
        "model": "Linear Regression - signal + core stock-structure features",
        "val_DA": val_DA_lr_sig_core,
        "test_DA": test_DA_lr_sig_core,
    },
    {
        "model": "XGBoost (tuned) - signal + core stock-structure features",
        "val_DA": val_DA_xgb_sig_core,
        "test_DA": test_DA_xgb_sig_core,
    },
    {
        "model": f"XGBoost (tuned) - {best_ablation['groups']}",
        "val_DA": best_ablation["val_DA"],
        "test_DA": best_ablation["test_DA"],
    },
    {
        "model": f"XGBoost (tuned) - {best_ablation_sig['label']}",
        "val_DA": best_ablation_sig["val_DA"],
        "test_DA": best_ablation_sig["test_DA"],
    },
]

results_models = pd.concat(
    [results_models, pd.DataFrame(new_rows)],
    ignore_index=True,
)

In [86]:

# Optional: sort by validation DA
results_models = results_models.sort_values("val_DA", ascending=False)
results_models

,model,val_DA,test_DA
0,XGBoost (tuned) - core stock structure features,0.516622,0.521387
8,XGBoost (tuned) - STRUCT + top1_signals,0.516261,0.514909
7,XGBoost (tuned) - dow+month+sector,0.514602,0.528145
6,XGBoost (tuned) - signal + core stock-structur...,0.513648,0.513922
1,XGBoost (tuned) - baseline features only,0.503996,0.518516
2,XGBoost (tuned) - top 15 signal baseline features,0.501615,0.522148
3,Linear Regression - baseline features only,0.486689,0.495691
5,Linear Regression - signal + core stock-struct...,0.485799,0.502898
4,Linear Regression - core stock structure features,0.480501,0.513169


#### Model 4 - 2 stage model (classifier then regression)

In [77]:
# ---------- 1. Cast features to float32 ----------
X_train_2s = X_train_sig_core_m.astype("float32")
X_val_2s   = X_val_sig_core_m.astype("float32")
X_test_2s  = X_test_sig_core_m.astype("float32")

# Make sure targets are 1D float32 arrays
y_train_arr = np.asarray(y_train_vec, dtype="float32").ravel()
y_val_arr   = np.asarray(y_val_vec,   dtype="float32").ravel()
y_test_arr  = np.asarray(y_test_vec,  dtype="float32").ravel()

# ---------- 2. Stage-1 labels: sign (classification target) ----------
# 1 = up, 0 = down-or-flat
y_train_sign = (y_train_arr > 0).astype(int)
y_val_sign   = (y_val_arr   > 0).astype(int)
y_test_sign  = (y_test_arr  > 0).astype(int)   # only for evaluation, not used in training logic

# ---------- 3. Stage-2 labels: magnitude ----------
y_train_mag = np.abs(y_train_arr)
y_val_mag   = np.abs(y_val_arr)
y_test_mag  = np.abs(y_test_arr)   # only for evaluation / inspection

In [78]:
# -------------------------------
# Stage 1: Classifier for sign
# -------------------------------
clf_xgb = xgb.XGBClassifier(
    max_depth=4,
    learning_rate=0.05,
    n_estimators=400,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
)

clf_xgb.fit(X_train_2s, y_train_sign)

# Predicted "probability of up" on val/test
p_val_up  = clf_xgb.predict_proba(X_val_2s)[:, 1]
p_test_up = clf_xgb.predict_proba(X_test_2s)[:, 1]

# ----- Threshold tuning on VAL to maximize DA -----
thresholds = np.linspace(0.3, 0.7, 41)  # search around 0.5
best_thr = 0.5
best_da_thr = -1.0

for thr in thresholds:
    sign_val_pred = np.where(p_val_up >= thr, 1.0, -1.0)
    da_thr = directional_accuracy(y_val_arr, sign_val_pred)
    if da_thr > best_da_thr:
        best_da_thr = da_thr
        best_thr = thr

print(f"[Stage 1] Best threshold on val: {best_thr:.3f} with Val DA = {best_da_thr:.4f}")

# Final sign predictions using best threshold
sign_val_pred  = np.where(p_val_up  >= best_thr, 1.0, -1.0)
sign_test_pred = np.where(p_test_up >= best_thr, 1.0, -1.0)

# (You can also inspect pure sign-model DA if you want)
print("Classifier-only Val DA:", directional_accuracy(y_val_arr,  sign_val_pred))
print("Classifier-only Test DA:", directional_accuracy(y_test_arr, sign_test_pred))

[Stage 1] Best threshold on val: 0.700 with Val DA = 0.5020
Classifier-only Val DA: 0.5019679831979927
Classifier-only Test DA: 0.4727043863798536


In [79]:
# -------------------------------
# Stage 2: Regressor for magnitude
# -------------------------------

# Reuse your tuned regression params from earlier
reg_xgb = xgb.XGBRegressor(
    **best_params,          # assumes best_params already defined from earlier tuning
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
)

reg_xgb.fit(X_train_2s, y_train_mag)

# Magnitude predictions on val/test
mag_val_pred  = reg_xgb.predict(X_val_2s)
mag_test_pred = reg_xgb.predict(X_test_2s)

# Make sure magnitudes are non-negative (just in case)
mag_val_pred  = np.abs(mag_val_pred)
mag_test_pred = np.abs(mag_test_pred)

In [80]:
# -------------------------------
# Combine: sign × magnitude
# -------------------------------

two_stage_val_pred  = sign_val_pred  * mag_val_pred
two_stage_test_pred = sign_test_pred * mag_test_pred

val_DA_2stage  = directional_accuracy(y_val_arr,  two_stage_val_pred)
test_DA_2stage = directional_accuracy(y_test_arr, two_stage_test_pred)

print("Two-stage model (sign × |return|):")
print("Val DA: ", val_DA_2stage)
print("Test DA:", test_DA_2stage)

Two-stage model (sign × |return|):
Val DA:  0.5019679831979927
Test DA: 0.4727043863798536


#### Model 5 - Add ticker fixed effects via embeddings **DON'T RUN AGAIN**

We experimented with a neural network with ticker embeddings. Despite early stopping and sign-aware training, its validation directional accuracy (~0.50) remained below that of our tuned gradient-boosted trees (~0.52), so we kept XGBoost as our primary model.

**Will NOT run again. Just noting that the results are worst than XGBoost**

In [84]:
feature_cols_all = [c for c in X_train_sig_core.columns if c not in ["Date", "tic"]]
X_train_sig_core_m = X_train_sig_core[feature_cols_all]
X_val_sig_core_m   = X_val_sig_core[feature_cols_all]
X_test_sig_core_m  = X_test_sig_core[feature_cols_all]

In [85]:
# 1) Build ticker vocabulary from *train only*
all_tickers_train = X_train_sig_core["tic"].unique()
ticker2id = {tic: i for i, tic in enumerate(sorted(all_tickers_train))}
num_tickers = len(ticker2id)
print("Num tickers:", num_tickers)

# Helper to map series of tic to integer ids
def tic_to_ids(tic_series):
    return tic_series.map(ticker2id).values

# 2) Numeric feature matrices (same as before)
feature_cols_all = [c for c in X_train_sig_core.columns if c not in ["Date", "tic"]]

X_train_num = X_train_sig_core[feature_cols_all].values.astype("float32")
X_val_num   = X_val_sig_core[feature_cols_all].values.astype("float32")
X_test_num  = X_test_sig_core[feature_cols_all].values.astype("float32")

# 3) Ticker id vectors
train_tic_ids = tic_to_ids(X_train_sig_core["tic"])
val_tic_ids   = tic_to_ids(X_val_sig_core["tic"])
test_tic_ids  = tic_to_ids(X_test_sig_core["tic"])

train_tic_ids = train_tic_ids.astype("int64")
val_tic_ids   = val_tic_ids.astype("int64")
test_tic_ids  = test_tic_ids.astype("int64")

# 4) Targets as float32
y_train_t = y_train_vec.values.astype("float32")
y_val_t   = y_val_vec.values.astype("float32")
y_test_t  = y_test_vec.values.astype("float32")

Num tickers: 496


/var/folders/c2/pprfn7j56vs360hbrz08yqkh0000gn/T/ipykernel_15319/834833684.py:24: RuntimeWarning: invalid value encountered in cast
  val_tic_ids   = val_tic_ids.astype("int64")
/var/folders/c2/pprfn7j56vs360hbrz08yqkh0000gn/T/ipykernel_15319/834833684.py:25: RuntimeWarning: invalid value encountered in cast
  test_tic_ids  = test_tic_ids.astype("int64")


In [86]:
from torch.utils.data import Dataset, DataLoader

class StockDataset(Dataset):
    def __init__(self, X_num, tic_ids, y):
        self.X_num = X_num
        self.tic_ids = tic_ids
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return {
            "x_num": torch.from_numpy(self.X_num[idx]),
            "tic_id": torch.tensor(self.tic_ids[idx], dtype=torch.long),
            "y": torch.tensor(self.y[idx], dtype=torch.float32),
        }

train_ds = StockDataset(X_train_num, train_tic_ids, y_train_t)
val_ds   = StockDataset(X_val_num,   val_tic_ids,   y_val_t)
test_ds  = StockDataset(X_test_num,  test_tic_ids,  y_test_t)

train_loader = DataLoader(train_ds, batch_size=2048, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=4096, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=2048, shuffle=False)

In [87]:
import torch.nn as nn
import torch.nn.functional as F

class TickerEmbeddingModel(nn.Module):
    def __init__(
        self,
        num_tickers,
        num_features,
        emb_dim=16,
        num_hidden_dim=64,
        joint_hidden_dim=64,
        joint_hidden_dim2=32,
        dropout_p=0.2,
    ):
        super().__init__()

        # --- Ticker embedding tower ---
        self.ticker_emb = nn.Embedding(num_embeddings=num_tickers, embedding_dim=emb_dim)
        self.tic_fc = nn.Linear(emb_dim, emb_dim)  # small projection

        # --- Numeric feature tower ---
        self.num_fc1 = nn.Linear(num_features, num_hidden_dim)
        self.num_fc2 = nn.Linear(num_hidden_dim, num_hidden_dim)

        # --- Joint tower ---
        joint_in_dim = num_hidden_dim + emb_dim
        self.joint_fc1 = nn.Linear(joint_in_dim, joint_hidden_dim)
        self.joint_fc2 = nn.Linear(joint_hidden_dim, joint_hidden_dim2)

        self.out = nn.Linear(joint_hidden_dim2, 1)

        self.dropout = nn.Dropout(dropout_p)
        self.bn_num = nn.BatchNorm1d(num_hidden_dim)

    def forward(self, x_num, tic_id):
        # Numeric tower
        x_num = F.relu(self.num_fc1(x_num))
        x_num = self.bn_num(x_num)
        x_num = F.relu(self.num_fc2(x_num))
        x_num = self.dropout(x_num)

        # Ticker tower
        emb = self.ticker_emb(tic_id)
        emb = F.relu(self.tic_fc(emb))

        # Joint tower
        x = torch.cat([x_num, emb], dim=1)
        x = F.relu(self.joint_fc1(x))
        x = self.dropout(x)
        x = F.relu(self.joint_fc2(x))
        x = self.dropout(x)

        out = self.out(x).squeeze(-1)
        return out

In [88]:
num_features = X_train_num.shape[1]

model = TickerEmbeddingModel(
    num_tickers=num_tickers,
    num_features=num_features,
    emb_dim=16,
    num_hidden_dim=64,
    joint_hidden_dim=64,
    joint_hidden_dim2=32,
    dropout_p=0.2,
)

In [89]:
len(train_ds), len(train_loader)

(729659, 357)

In [90]:
def sign_aware_loss(pred, y, lambda_sign=0.1):
    mse = F.mse_loss(pred, y)

    # sign mismatch penalty: 1 if wrong sign, 0 if correct
    sign_mismatch = (torch.sign(pred) != torch.sign(y)).float()

    # average mismatch (fraction wrong)
    sign_penalty = sign_mismatch.mean()

    return mse + lambda_sign * sign_penalty

In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam

# -------------------------
# 1) Force CPU (avoid MPS / CUDA crashes)
# -------------------------
device = torch.device("cpu")
print("Using CPU")

model = model.to(device)

optimizer = Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)
loss_fn = lambda pred, y: sign_aware_loss(pred, y, lambda_sign=0.05)

def directional_accuracy_torch(y_true, y_pred):
    """
    y_true, y_pred: torch tensors, shape (batch,)
    Returns DA as a Python float.
    """
    return (torch.sign(y_true) == torch.sign(y_pred)).float().mean().item()

# -------------------------
# 2) Training loop with fewer epochs + early stopping on Val DA
# -------------------------
num_epochs = 20          
patience   = 10          # stop if no DA improv. for 5 epochs
best_val_da = -1.0
patience_counter = 0
best_state = None

for epoch in range(num_epochs):
    # ---- Train ----
    model.train()
    train_loss = 0.0

    for batch in train_loader:
        x_num = batch["x_num"].to(device)
        tic_id = batch["tic_id"].to(device)
        y = batch["y"].to(device)

        optimizer.zero_grad()
        pred = model(x_num, tic_id)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * len(y)

    train_loss /= len(train_ds)

    # ---- Validation ----
    model.eval()
    val_loss = 0.0
    val_da_sum = 0.0
    n_val = 0

    with torch.no_grad():
        for batch in val_loader:
            x_num = batch["x_num"].to(device)
            tic_id = batch["tic_id"].to(device)
            y = batch["y"].to(device)

            pred = model(x_num, tic_id)
            loss = loss_fn(pred, y)

            val_loss += loss.item() * len(y)
            batch_da = directional_accuracy_torch(y, pred)
            val_da_sum += batch_da * len(y)
            n_val += len(y)

    val_loss /= n_val
    val_da = val_da_sum / n_val

    print(
        f"Epoch {epoch+1:02d} | "
        f"train_loss={train_loss:.6f} | "
        f"val_loss={val_loss:.6f} | "
        f"val_DA={val_da:.4f}"
    )

    # ---- Early stopping on Val DA ----
    if val_da > best_val_da + 1e-4:
        best_val_da = val_da
        best_state = model.state_dict()
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

# Restore best model (optional but recommended)
if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Restored best model with Val DA = {best_val_da:.4f}")

Using CPU


In [ ]:
def predict_loader(model, loader):
    model.eval()
    preds = []
    ys    = []
    with torch.no_grad():
        for batch in loader:
            x_num = batch["x_num"].to(device)
            tic_id = batch["tic_id"].to(device)
            y = batch["y"].to(device)

            pred = model(x_num, tic_id)
            preds.append(pred.cpu().numpy())
            ys.append(y.cpu().numpy())

    preds = np.concatenate(preds)
    ys    = np.concatenate(ys)
    return ys, preds

y_val_true_emb, y_val_pred_emb   = predict_loader(model, val_loader)
y_test_true_emb, y_test_pred_emb = predict_loader(model, test_loader)

val_DA_emb  = (np.sign(y_val_pred_emb)  == np.sign(y_val_true_emb)).mean()
test_DA_emb = (np.sign(y_test_pred_emb) == np.sign(y_test_true_emb)).mean()

print("Ticker-Embedding Model:")
print("Val DA: ", val_DA_emb)
print("Test DA:", test_DA_emb)

Ticker-Embedding Model:
Val DA:  0.4930539411769421
Test DA: 0.5198553641152277


#### Model 6 - CAT boost

In [91]:
from catboost import CatBoostRegressor

# 1. Choose only numeric columns (drop Date and tic)
num_cols = [c for c in X_train_sig_core.columns if c not in ["Date", "tic"]]

X_train_cb = X_train_sig_core[num_cols]
X_val_cb   = X_val_sig_core[num_cols]
X_test_cb  = X_test_sig_core[num_cols]

# 2. Define CatBoost model
cat_model = CatBoostRegressor(
    depth=6,
    learning_rate=0.03,
    iterations=800,
    loss_function="RMSE",
    random_seed=42,
    verbose=False
)

# 3. Fit on train → validate on val
cat_model.fit(
    X_train_cb,
    y_train_vec,
    eval_set=(X_val_cb, y_val_vec),
    verbose=False
)

# 4. Predictions + DA
pred_val_cat  = cat_model.predict(X_val_cb)
pred_test_cat = cat_model.predict(X_test_cb)

val_DA_cat  = directional_accuracy(y_val_vec,  pred_val_cat)
test_DA_cat = directional_accuracy(y_test_vec, pred_test_cat)

print("CatBoost (sig + core, numeric only):")
print("Val DA: ", val_DA_cat)
print("Test DA:", test_DA_cat)

CatBoost (sig + core, numeric only):
Val DA:  0.4929096491298388
Test DA: 0.5198553641152277


In [92]:
from catboost import CatBoostRegressor

# Use all columns except Date (keep tic as categorical)
feature_cols_cb = [c for c in X_train_sig_core.columns if c != "Date"]

X_train_cb = X_train_sig_core[feature_cols_cb]
X_val_cb   = X_val_sig_core[feature_cols_cb]
X_test_cb  = X_test_sig_core[feature_cols_cb]

# Cat features: columns that are strings / categories
cat_features = [feature_cols_cb.index("tic")]  # index of 'tic' in feature_cols_cb

cat_model = CatBoostRegressor(
    depth=6,
    learning_rate=0.03,
    iterations=800,
    loss_function="RMSE",
    random_seed=42,
    verbose=False
)

cat_model.fit(
    X_train_cb,
    y_train_vec,
    eval_set=(X_val_cb, y_val_vec),
    cat_features=cat_features,
    verbose=False
)

pred_val_cat  = cat_model.predict(X_val_cb)
pred_test_cat = cat_model.predict(X_test_cb)

val_DA_cat  = directional_accuracy(y_val_vec,  pred_val_cat)
test_DA_cat = directional_accuracy(y_test_vec, pred_test_cat)

print("CatBoost (tic as categorical):")
print("Val DA: ", val_DA_cat)
print("Test DA:", test_DA_cat)

CatBoost (tic as categorical):
Val DA:  0.4917312640784949
Test DA: 0.5181797044745725


In [ ]:
from lightgbm import LGBMRegressor

lgbm = LGBMRegressor(
    n_estimators=1500,
    max_depth=-1,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
)

lgbm.fit(X_train_sig_core_m, y_train_vec)

pred_val_lgbm = lgbm.predict(X_val_sig_core_m)
pred_test_lgbm = lgbm.predict(X_test_sig_core_m)